# Module 24 — The AdamW Optimizer

Modules 03/05 used plain SGD: every parameter gets nudged by exactly
`learning_rate * gradient`, regardless of how that gradient behaves over
time. **Adam** tracks, per parameter, a running average of the gradient
(momentum, "first moment") and a running average of the *squared* gradient
("second moment"), then divides the step by roughly the gradient's typical
size — parameters with consistently large gradients get smaller effective
steps, and vice versa. This adapts well to how differently-scaled
gradients are across a deep transformer's many parameters, which is why
Adam (and specifically AdamW) is the near-universal choice for training
them, not SGD.

## 1. Adam from scratch, verified against `torch.optim.Adam`

`weight_decay` in plain Adam is applied the old way: L2 regularization
folded directly into the gradient (`g = grad + weight_decay * param`)
*before* the adaptive scaling happens.

In [ ]:
import copy

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
model_ref = nn.Sequential(nn.Linear(8, 8), nn.ReLU(), nn.Linear(8, 4))
model_scratch = copy.deepcopy(model_ref)

x = torch.randn(6, 8)
y = torch.randint(0, 4, (6,))
lr, beta1, beta2, eps, wd = 0.01, 0.9, 0.999, 1e-8, 0.01

opt_ref = torch.optim.Adam(model_ref.parameters(), lr=lr, betas=(beta1, beta2), eps=eps, weight_decay=wd)
state = [{"m": torch.zeros_like(p), "v": torch.zeros_like(p)} for p in model_scratch.parameters()]

for step in range(1, 6):
    opt_ref.zero_grad()
    F.cross_entropy(model_ref(x), y).backward()
    opt_ref.step()

    for p in model_scratch.parameters():
        p.grad = None
    F.cross_entropy(model_scratch(x), y).backward()
    with torch.no_grad():
        for p, s in zip(model_scratch.parameters(), state):
            g = p.grad + wd * p  # L2 regularization folded into the gradient (plain Adam)
            s["m"] = beta1 * s["m"] + (1 - beta1) * g
            s["v"] = beta2 * s["v"] + (1 - beta2) * g * g
            m_hat = s["m"] / (1 - beta1 ** step)
            v_hat = s["v"] / (1 - beta2 ** step)
            p -= lr * m_hat / (v_hat.sqrt() + eps)

for p_ref, p_scratch in zip(model_ref.parameters(), model_scratch.parameters()):
    assert torch.allclose(p_ref, p_scratch, atol=1e-6)
print("From-scratch Adam matches torch.optim.Adam(weight_decay=...) exactly.")

## 2. AdamW: the same adaptive update, but *decoupled* weight decay

AdamW applies decay directly to the parameter — `p -= lr * weight_decay *
p` — as its own separate step, computed from the raw gradient (not
folded into `m`/`v` at all). Matching PyTorch's actual implementation
exactly requires getting the order right: decay is subtracted from the
parameter **first**, then the Adam moment update runs on top of that
already-decayed value.

In [ ]:
torch.manual_seed(0)  # fresh init, independent comparison from the plain-Adam run above
model_ref_w = nn.Sequential(nn.Linear(8, 8), nn.ReLU(), nn.Linear(8, 4))
model_scratch_w = copy.deepcopy(model_ref_w)

opt_ref_w = torch.optim.AdamW(model_ref_w.parameters(), lr=lr, betas=(beta1, beta2), eps=eps, weight_decay=wd)
state_w = [{"m": torch.zeros_like(p), "v": torch.zeros_like(p)} for p in model_scratch_w.parameters()]

for step in range(1, 6):
    opt_ref_w.zero_grad()
    F.cross_entropy(model_ref_w(x), y).backward()
    opt_ref_w.step()

    for p in model_scratch_w.parameters():
        p.grad = None
    F.cross_entropy(model_scratch_w(x), y).backward()
    with torch.no_grad():
        for p, s in zip(model_scratch_w.parameters(), state_w):
            g = p.grad  # NOT combined with weight decay
            p -= lr * wd * p  # decoupled decay, applied directly, first
            s["m"] = beta1 * s["m"] + (1 - beta1) * g
            s["v"] = beta2 * s["v"] + (1 - beta2) * g * g
            m_hat = s["m"] / (1 - beta1 ** step)
            v_hat = s["v"] / (1 - beta2 ** step)
            p -= lr * m_hat / (v_hat.sqrt() + eps)

for p_ref, p_scratch in zip(model_ref_w.parameters(), model_scratch_w.parameters()):
    assert torch.allclose(p_ref, p_scratch, atol=1e-6)
print("From-scratch AdamW matches torch.optim.AdamW exactly (and needed the decay-first ordering to do it).")

## 3. Why decoupling matters: L2-via-gradient decays inconsistently

Here's the concrete problem AdamW fixes. Under plain Adam, the weight
decay term gets folded into the gradient, which means it gets divided by
`sqrt(v_hat)` exactly like the real gradient signal does. A parameter
that's been seeing **large** gradients (big `v_hat`) has its decay
*dampened*; a parameter seeing **small** gradients (tiny `v_hat`) has its
decay *amplified* — for the exact same `weight_decay` setting. That's
backwards from what "decay every weight by the same proportion" is
supposed to mean.

In [ ]:
p_value = 1.0
lr_demo, wd_demo = 0.1, 0.01

v_hat_large = 100.0    # a parameter with a history of large gradients
v_hat_small = 0.0001   # a parameter with a history of tiny gradients

# Adam-L2: the decay contribution passes through the SAME adaptive division as the real gradient
adam_l2_decay_largev = lr_demo * wd_demo * p_value / (v_hat_large ** 0.5)
adam_l2_decay_smallv = lr_demo * wd_demo * p_value / (v_hat_small ** 0.5)

# AdamW: decay is applied directly to the parameter, independent of v_hat entirely
adamw_decay = lr_demo * wd_demo * p_value

print(f"Adam-L2 decay amount, parameter with large historical gradients: {adam_l2_decay_largev:.6f}")
print(f"Adam-L2 decay amount, parameter with small historical gradients: {adam_l2_decay_smallv:.6f}")
print(f"  -> {adam_l2_decay_smallv / adam_l2_decay_largev:.0f}x more decay for the small-gradient parameter, for the SAME weight_decay setting")
print(f"\nAdamW decay amount, either parameter: {adamw_decay:.6f}  (identical, regardless of gradient history)")

assert adam_l2_decay_smallv > adam_l2_decay_largev * 100
print("\nConfirmed: L2-via-gradient makes decay strength depend on gradient history; AdamW makes it consistent.")

## Recap

- Adam's adaptive per-parameter step size was verified exactly against
  `torch.optim.Adam`, and AdamW's decoupled weight decay against
  `torch.optim.AdamW` — both matched to float precision.
- The concrete reason AdamW exists: folding weight decay into the gradient
  (plain Adam's approach) makes the *actual* decay strength depend on each
  parameter's gradient history — demonstrated directly with a 1,000,000x
  discrepancy between a high-gradient-history and low-gradient-history
  parameter under identical settings. AdamW's decoupled decay is applied
  uniformly, independent of gradient history, which is what "weight decay"
  is supposed to mean.
- This is why every transformer this project builds toward — nanoGPT
  onward — uses AdamW, not plain SGD or plain Adam.

Module 25 covers learning rate schedules — warmup and cosine decay — the
other standard piece of a real training configuration.